# Возможности `zemi.toml`

Модуль возвращает стандартное дерево `dict`/`list`, проверяет уникальность имён и существование ZEMI-ссылок, но не раскрывает их содержимое.

In [ ]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / '.zemicomp').is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from zemi import env, toml

CONFIG_PATH = PROJECT_ROOT / 'tests/zemi_toml/test_zemi_toml.toml'
config = toml.load(CONFIG_PATH)

## Обычное дерево Python

Вложенные таблицы остаются словарями, а массивы таблиц — списками. Предметное именованное дерево создаёт использующий конфигурацию модуль, например `zemi.playbook.arsenal`.

In [ ]:
assert type(config) is dict
assert type(config['arsenal']) is dict
assert type(config['arsenal']['llamas']) is list
primary = config['arsenal']['llamas'][0]
qwen = primary['models'][0]
assistant = qwen['assistants'][0]
primary['name'], qwen['name'], assistant['name']

## ZEMI-ссылки сохраняются

`@comp/...` и `@inst/...` проверяются на существование, но остаются исходными строками. Содержимое файла читает только его непосредственный потребитель.

In [ ]:
prefix = assistant['prefix']
assert prefix == '@comp/tests/zemi_toml/prefixes/qwen-system.md'
assert config['non_text_reference'] == '@comp/runme.toml'
prefix

## Валидация

Повторяющиеся непустые `name` в одном массиве и ссылки на отсутствующие пути отклоняются. Временные файлы создаются только в `@inst/_tmp`.

In [ ]:
env.path.tmp.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=env.path.tmp) as directory:
    duplicate = Path(directory) / 'duplicate.toml'
    duplicate.write_text(
        "[[items]]\nname = 'same'\n[[items]]\nname = 'same'\n",
        encoding='utf-8',
    )
    try:
        toml.load(duplicate)
    except ValueError as error:
        assert "повторяющееся имя 'same'" in str(error)
    else:
        raise AssertionError('Ожидался ValueError')

with TemporaryDirectory(dir=env.path.tmp) as directory:
    missing = Path(directory) / 'missing.toml'
    missing.write_text(
        "reference = '@comp/does-not-exist-for-toml-test.txt'\n",
        encoding='utf-8',
    )
    try:
        toml.load(missing)
    except FileNotFoundError:
        pass
    else:
        raise AssertionError('Ожидался FileNotFoundError')